# 02 — Backtest EMACross

Run our EMACross strategy against BTC-USD-PERP 1h data on a simulated Hyperliquid venue.

**Sections:**
1. Setup — imports, catalog, engine configuration
2. Single run — fills, positions, account reports
3. Visualisation — price + EMA overlay with trade markers, equity curve
4. Statistics — analyzer performance summary
5. Parameter sweep — fast/slow grid with heatmap

**Prerequisites:** Run `scripts/fetch_hl_candles.py` first to populate `data/catalog/`.

In [ ]:
# ── Cell 1: Imports + shared config ────────────────────────────────
#
# All tuneable values live here. Change once, used everywhere below.

from decimal import Decimal

import pandas as pd
from IPython.display import HTML

from nautilus_trader.analysis import create_tearsheet
from nautilus_trader.model.currencies import USDC
from nautilus_trader.model.data import BarType
from nautilus_trader.model.identifiers import Venue
from nautilus_trader.persistence.catalog import ParquetDataCatalog

from src.backtesting import make_engine, run_single_backtest, run_sweep
from src.core import bar_type_str
from src.strategies.ema_cross import EMACross, EMACrossConfig

from charts import plot_ema_cross, plot_equity_curve, print_summary_stats, plot_pnl_heatmap, generate_backtest_html

# ── Shared config ────────────────────────────────────────────────
CATALOG_PATH    = "../data/catalog"
INSTRUMENT_ID   = "BTC-USD-PERP.HYPERLIQUID"
BAR_INTERVAL    = "1h"
BAR_TYPE_STR    = bar_type_str(INSTRUMENT_ID, BAR_INTERVAL)
VENUE           = Venue("HYPERLIQUID")
STARTING_CAPITAL = 10_000
TRADE_SIZE      = Decimal("0.02")   # 0.01 BTC per trade (~$650 notional at $65k)
FAST_EMA        = 20
SLOW_EMA        = 50
FAST_PERIODS = sorted(set([5, 10, 15, 20, 25, 30, 35, 40, 45, 50, 75, 100] + [FAST_EMA]))
SLOW_PERIODS = sorted(set([10, 15, 20, 25, 30, 35, 40, 45, 50, 75, 100, 200] + [SLOW_EMA]))

print("Imports OK")

In [ ]:
# ── Cell 2: Load data from catalog ────────────────────────────────
catalog    = ParquetDataCatalog(CATALOG_PATH)
instrument = catalog.instruments(instrument_ids=[INSTRUMENT_ID])[0]
bars       = catalog.bars(bar_types=[BAR_TYPE_STR])

print(f"Instrument : {instrument.id}")
print(f"Bar count  : {len(bars):,}")
print(f"First bar  : {pd.Timestamp(bars[0].ts_event, unit='ns', tz='UTC')}")
print(f"Last bar   : {pd.Timestamp(bars[-1].ts_event, unit='ns', tz='UTC')}")

In [ ]:
# ── Cell 3: Configure engine + venue ──────────────────────────────
engine = make_engine(VENUE, instrument, bars, STARTING_CAPITAL)
print("Engine configured.")

In [ ]:
# ── Cell 4: Add EMACross strategy + run ───────────────────────────
config = EMACrossConfig(
    instrument_id=instrument.id,
    bar_type=BarType.from_str(BAR_TYPE_STR),
    trade_size=TRADE_SIZE,
    fast_ema_period=FAST_EMA,
    slow_ema_period=SLOW_EMA,
)
strategy = EMACross(config=config)
engine.add_strategy(strategy)
engine.run()
print("Backtest complete.")

In [ ]:
# ── Cell 5: Reports ───────────────────────────────────────────────
fills_report     = engine.trader.generate_order_fills_report()
positions_report = engine.trader.generate_positions_report()
account_report   = engine.trader.generate_account_report(VENUE)

print(f"Order fills : {len(fills_report)}")
print(f"Positions   : {len(positions_report)}")
print()

print("=== Recent Fills ===")
display(fills_report.tail(10))

print("\n=== Recent Positions ===")
display(positions_report.tail(10))

In [ ]:
# ── Cell 6: Calculate statistics ──────────────────────────────────
#
# Must run before equity curve — analyzer.returns() is a getter that
# only has data after calculate_statistics() populates it.

account   = engine.cache.account_for_venue(VENUE)
positions = engine.cache.position_snapshots() + engine.cache.positions()
analyzer  = engine.portfolio.analyzer
analyzer.calculate_statistics(account, positions)
print(f"Stats calculated — {len(positions)} positions")

In [ ]:
# ── Cell 7: HTML Tearsheet ────────────────────────────────────────
html = create_tearsheet(
    engine,
    output_path=None,
    title=f"EMACross({FAST_EMA}/{SLOW_EMA}) | {INSTRUMENT_ID} {BAR_INTERVAL}",
)
display(HTML(html))

In [ ]:
# ── Cell 8: Price chart with EMA overlay + trade markers ──────────
#
# Recompute EMA values from bars for plotting (the strategy's internal
# indicator state isn't directly accessible after the run).

fig = plot_ema_cross(
    bars, fills_report,
    fast_period=FAST_EMA,
    slow_period=SLOW_EMA,
    instrument_label=INSTRUMENT_ID,
    bar_label=BAR_INTERVAL,
)
fig.show(config=dict(
    modeBarButtonsToRemove=["autoScale2d", "lasso2d", "select2d"],
    displaylogo=False,
))

In [ ]:
# ── Cell 9: Equity curve ──────────────────────────────────────────
plot_equity_curve(analyzer, account_report, f"EMACross({FAST_EMA}/{SLOW_EMA})  {INSTRUMENT_ID} {BAR_INTERVAL}")

In [ ]:
# ── Cell 10: Summary statistics ────────────────────────────────────
print_summary_stats(analyzer, num_positions=len(positions))

In [ ]:
# ── Cell 11: Parameter sweep ──────────────────────────────────────
#
# Grid search over fast/slow EMA period combinations.

def ema_factory(eng, params):
    cfg = EMACrossConfig(
        instrument_id=instrument.id,
        bar_type=BarType.from_str(BAR_TYPE_STR),
        trade_size=TRADE_SIZE,
        fast_ema_period=params["fast"],
        slow_ema_period=params["slow"],
    )
    eng.add_strategy(EMACross(cfg))

combos = [{"fast": f, "slow": s} for f in FAST_PERIODS for s in SLOW_PERIODS if f < s]

results_df = run_sweep(
    venue=VENUE, instrument=instrument, bars=bars,
    starting_capital=STARTING_CAPITAL,
    param_combos=combos,
    strategy_factory=ema_factory,
    strategy_name="EMACross",
    instrument_id=INSTRUMENT_ID,
    bar_interval=BAR_INTERVAL,
)

In [ ]:
# ── Cell 12: PnL heatmap ──────────────────────────────────────────
plot_pnl_heatmap(
    results_df, row_col="slow", col_col="fast",
    row_label="Slow EMA Period", col_label="Fast EMA Period",
    title="Total PnL (USDC) by EMA Parameters",
)

In [ ]:
# ── Cell 13: TradingView Interactive Backtest ──────────────────────
path = generate_backtest_html(
    bars, fills_report, positions_report,
    fast_period=FAST_EMA,
    slow_period=SLOW_EMA,
    instrument_label=INSTRUMENT_ID,
    bar_label=BAR_INTERVAL,
    starting_capital=STARTING_CAPITAL,
    output_path=f"../reports/ema_cross_{FAST_EMA}_{SLOW_EMA}_{BAR_INTERVAL}.html",
    open_browser=True,
)

In [ ]:
# ── Cell 14: Cleanup ──────────────────────────────────────────────
engine.dispose()
print("Engine disposed.")